# Business Scenario Analysis — Fraud Detection

The same model can serve very different business needs depending on how you tune the decision threshold.
This notebook walks through two realistic scenarios with concrete cost assumptions and shows how
the optimal threshold, metrics, and confusion matrix change for each.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, f1_score, fbeta_score,
    precision_score, recall_score
)
import joblib
import os

sns.set_style('whitegrid')
%matplotlib inline

## Load Model & Test Data

We use the same trained model for both scenarios — only the threshold changes.

In [ ]:
# Update these paths if your model is in a different subfolder
MODEL_PATH = '../models/train_smote/XGBoost.joblib'
DATA_DIR = '../data'

model = joblib.load(MODEL_PATH)
X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

y_scores = model.predict_proba(X_test)[:, 1]

print(f'Model: {type(model).__name__}')
print(f'Test set: {len(y_test):,} transactions ({y_test.sum():.0f} fraud)')

---

## Scenario A: Large Retail Bank with Manual Review Team

### Business context

| Assumption | Value |
|---|---|
| Average loss per undetected fraud (FN) | €120 |
| Cost per manual review (each flagged txn) | €2 |
| Customer friction from review call | Low — most appreciate the check |
| Review team capacity | Large, already staffed |

### What this means

Missing a fraud costs **60x** more than a false alarm. The bank can absorb a high volume of false positives
because each one only triggers a quick analyst call. The priority is **catching as much fraud as possible**.

### Optimization strategy

- Optimize for **high recall** (catch rate)
- Use **F2 score** (weights recall 2x over precision) or target a minimum recall of 95%
- Accept more false positives — they're cheap

In [ ]:
# Scenario A cost parameters
COST_FN_A = 120   # cost of missing a fraud
COST_FP_A = 2     # cost of a false alarm (manual review)
COST_TP_A = 2     # review cost even for correctly caught fraud
COST_TN_A = 0     # correctly passed legitimate transaction


def compute_total_cost(y_true, y_pred, cost_fn, cost_fp, cost_tp=0, cost_tn=0):
    """Compute total business cost from a confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total = (tn * cost_tn) + (fp * cost_fp) + (fn * cost_fn) + (tp * cost_tp)
    return total, {'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp}


# Find threshold that minimizes total cost for Scenario A
thresholds = np.arange(0.01, 0.99, 0.01)
costs_a = []

for t in thresholds:
    y_pred = (y_scores >= t).astype(int)
    cost, _ = compute_total_cost(y_test, y_pred, COST_FN_A, COST_FP_A, COST_TP_A)
    costs_a.append(cost)

best_idx_a = np.argmin(costs_a)
best_threshold_a = thresholds[best_idx_a]
best_cost_a = costs_a[best_idx_a]

print(f'Optimal threshold (min cost): {best_threshold_a:.2f}')
print(f'Minimum total cost: EUR {best_cost_a:,.0f}')

In [ ]:
# Also find F2-optimal and recall-95% thresholds for comparison
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_scores)

f2_scores = 5 * (precisions[:-1] * recalls[:-1]) / (4 * precisions[:-1] + recalls[:-1] + 1e-8)
best_f2_idx = np.argmax(f2_scores)
threshold_f2_a = pr_thresholds[best_f2_idx]

# Recall >= 95%: among qualifying thresholds, pick highest precision
valid_95 = recalls[:-1] >= 0.95
if valid_95.any():
    candidates = np.where(valid_95)[0]
    best_r95_idx = candidates[np.argmax(precisions[:-1][candidates])]
    threshold_r95_a = pr_thresholds[best_r95_idx]
else:
    threshold_r95_a = 0.1

print(f'F2-optimal threshold:   {threshold_f2_a:.4f}')
print(f'Recall>=95% threshold:  {threshold_r95_a:.4f}')
print(f'Cost-optimal threshold: {best_threshold_a:.4f}')

In [ ]:
# Scenario A: evaluate at cost-optimal threshold
y_pred_a = (y_scores >= best_threshold_a).astype(int)
cost_a, cm_a = compute_total_cost(y_test, y_pred_a, COST_FN_A, COST_FP_A, COST_TP_A)

print(f'=== Scenario A: Retail Bank (threshold={best_threshold_a:.2f}) ===')
print(classification_report(y_test, y_pred_a, target_names=['Legit', 'Fraud'], digits=4))
print(f"Fraud caught:   {cm_a['tp']} / {cm_a['tp'] + cm_a['fn']} ({cm_a['tp']/(cm_a['tp']+cm_a['fn']):.1%})")
print(f"False alarms:   {cm_a['fp']:,}")
print(f"Missed fraud:   {cm_a['fn']}")
print(f"Total cost:     EUR {cost_a:,.0f}")
print(f"\nCost breakdown:")
print(f"  Missed fraud:  EUR {cm_a['fn'] * COST_FN_A:,.0f}  ({cm_a['fn']} x EUR {COST_FN_A})")
print(f"  Reviews:       EUR {(cm_a['fp'] + cm_a['tp']) * COST_FP_A:,.0f}  ({cm_a['fp'] + cm_a['tp']:,} x EUR {COST_FP_A})")

In [ ]:
# Scenario A: cost curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Cost vs threshold
ax1.plot(thresholds, costs_a, 'b-', linewidth=2)
ax1.axvline(x=best_threshold_a, color='red', linestyle='--',
            label=f'Optimal: {best_threshold_a:.2f} (EUR {best_cost_a:,.0f})')
ax1.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='Default 0.5')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Total Business Cost (EUR)', fontsize=12)
ax1.set_title('Scenario A: Cost vs Threshold', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Confusion matrix
cm_matrix_a = confusion_matrix(y_test, y_pred_a)
sns.heatmap(cm_matrix_a, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
ax2.set_title(f'Scenario A: Confusion Matrix (t={best_threshold_a:.2f})', fontsize=14)
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---

## Scenario B: Fintech with Automated Transaction Blocking

### Business context

| Assumption | Value |
|---|---|
| Average loss per undetected fraud (FN) | €80 |
| Cost per false decline (FP) | €15 (support ticket) + €200 (churn risk / LTV loss) = €215 |
| Review process | None — flagged transactions are **instantly declined** |
| Customer impact | High — declined purchase triggers frustration and churn |

### What this means

A false positive costs **~2.7x** more than a missed fraud. There's no human in the loop to catch mistakes —
every flag is an automatic block. The priority is **high precision**: only decline when very confident.

### Optimization strategy

- Optimize for **high precision** (low false alarm rate)
- Use **F0.5 score** (weights precision 2x over recall) or a cost-minimizing threshold
- Accept that some fraud slips through — it's cheaper than alienating customers

In [ ]:
# Scenario B cost parameters
COST_FN_B = 80    # cost of missing a fraud
COST_FP_B = 215   # cost of a false decline (support + churn)
COST_TP_B = 0     # correctly blocked fraud — no cost
COST_TN_B = 0     # correctly passed legitimate

costs_b = []

for t in thresholds:
    y_pred = (y_scores >= t).astype(int)
    cost, _ = compute_total_cost(y_test, y_pred, COST_FN_B, COST_FP_B, COST_TP_B)
    costs_b.append(cost)

best_idx_b = np.argmin(costs_b)
best_threshold_b = thresholds[best_idx_b]
best_cost_b = costs_b[best_idx_b]

print(f'Optimal threshold (min cost): {best_threshold_b:.2f}')
print(f'Minimum total cost: EUR {best_cost_b:,.0f}')

In [ ]:
# F0.5 score — weights precision higher
f05_scores = (1 + 0.25) * (precisions[:-1] * recalls[:-1]) / (0.25 * precisions[:-1] + recalls[:-1] + 1e-8)
best_f05_idx = np.argmax(f05_scores)
threshold_f05_b = pr_thresholds[best_f05_idx]

print(f'F0.5-optimal threshold: {threshold_f05_b:.4f}')
print(f'Cost-optimal threshold: {best_threshold_b:.4f}')

In [ ]:
# Scenario B: evaluate at cost-optimal threshold
y_pred_b = (y_scores >= best_threshold_b).astype(int)
cost_b, cm_b = compute_total_cost(y_test, y_pred_b, COST_FN_B, COST_FP_B, COST_TP_B)

print(f'=== Scenario B: Fintech (threshold={best_threshold_b:.2f}) ===')
print(classification_report(y_test, y_pred_b, target_names=['Legit', 'Fraud'], digits=4))
print(f"Fraud caught:   {cm_b['tp']} / {cm_b['tp'] + cm_b['fn']} ({cm_b['tp']/(cm_b['tp']+cm_b['fn']):.1%})")
print(f"False declines: {cm_b['fp']:,}")
print(f"Missed fraud:   {cm_b['fn']}")
print(f"Total cost:     EUR {cost_b:,.0f}")
print(f"\nCost breakdown:")
print(f"  Missed fraud:    EUR {cm_b['fn'] * COST_FN_B:,.0f}  ({cm_b['fn']} x EUR {COST_FN_B})")
print(f"  False declines:  EUR {cm_b['fp'] * COST_FP_B:,.0f}  ({cm_b['fp']:,} x EUR {COST_FP_B})")

In [ ]:
# Scenario B: cost curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(thresholds, costs_b, 'r-', linewidth=2)
ax1.axvline(x=best_threshold_b, color='red', linestyle='--',
            label=f'Optimal: {best_threshold_b:.2f} (EUR {best_cost_b:,.0f})')
ax1.axvline(x=0.5, color='gray', linestyle=':', alpha=0.7, label='Default 0.5')
ax1.set_xlabel('Decision Threshold', fontsize=12)
ax1.set_ylabel('Total Business Cost (EUR)', fontsize=12)
ax1.set_title('Scenario B: Cost vs Threshold', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

cm_matrix_b = confusion_matrix(y_test, y_pred_b)
sns.heatmap(cm_matrix_b, annot=True, fmt='d', cmap='Reds', ax=ax2,
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
ax2.set_title(f'Scenario B: Confusion Matrix (t={best_threshold_b:.2f})', fontsize=14)
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

---

## Side-by-Side Comparison

In [ ]:
# Side-by-side cost curves
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Cost curves overlaid
ax = axes[0]
ax.plot(thresholds, costs_a, 'b-', linewidth=2, label='A: Bank')
ax.plot(thresholds, costs_b, 'r-', linewidth=2, label='B: Fintech')
ax.axvline(x=best_threshold_a, color='blue', linestyle='--', alpha=0.7,
           label=f'A optimal: {best_threshold_a:.2f}')
ax.axvline(x=best_threshold_b, color='red', linestyle='--', alpha=0.7,
           label=f'B optimal: {best_threshold_b:.2f}')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Total Business Cost (EUR)', fontsize=12)
ax.set_title('Cost Curves: Both Scenarios', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Confusion matrices side by side
sns.heatmap(cm_matrix_a, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[1].set_title(f'A: Bank (t={best_threshold_a:.2f})\nHigh recall, more false alarms', fontsize=11)
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

sns.heatmap(cm_matrix_b, annot=True, fmt='d', cmap='Reds', ax=axes[2],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[2].set_title(f'B: Fintech (t={best_threshold_b:.2f})\nHigh precision, some missed fraud', fontsize=11)
axes[2].set_ylabel('Actual')
axes[2].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Metric': [
        'Threshold',
        'Precision',
        'Recall',
        'F1',
        'F0.5 (precision-weighted)',
        'F2 (recall-weighted)',
        'False Positives',
        'False Negatives (missed fraud)',
        'Total Cost (EUR)',
        'Cost per transaction (EUR)',
    ],
    'A: Retail Bank': [
        f'{best_threshold_a:.2f}',
        f'{precision_score(y_test, y_pred_a):.4f}',
        f'{recall_score(y_test, y_pred_a):.4f}',
        f'{f1_score(y_test, y_pred_a):.4f}',
        f'{fbeta_score(y_test, y_pred_a, beta=0.5):.4f}',
        f'{fbeta_score(y_test, y_pred_a, beta=2):.4f}',
        f'{cm_a["fp"]:,}',
        f'{cm_a["fn"]}',
        f'{cost_a:,.0f}',
        f'{cost_a / len(y_test):.4f}',
    ],
    'B: Fintech': [
        f'{best_threshold_b:.2f}',
        f'{precision_score(y_test, y_pred_b):.4f}',
        f'{recall_score(y_test, y_pred_b):.4f}',
        f'{f1_score(y_test, y_pred_b):.4f}',
        f'{fbeta_score(y_test, y_pred_b, beta=0.5):.4f}',
        f'{fbeta_score(y_test, y_pred_b, beta=2):.4f}',
        f'{cm_b["fp"]:,}',
        f'{cm_b["fn"]}',
        f'{cost_b:,.0f}',
        f'{cost_b / len(y_test):.4f}',
    ],
})

print('=== Side-by-Side Comparison ===')
print()
print(comparison.to_string(index=False))

In [ ]:
# What if each scenario used the OTHER's threshold?
print('=== What happens with the wrong threshold? ===\n')

# Bank using fintech threshold (too high)
y_pred_a_wrong = (y_scores >= best_threshold_b).astype(int)
cost_a_wrong, cm_a_wrong = compute_total_cost(y_test, y_pred_a_wrong, COST_FN_A, COST_FP_A, COST_TP_A)
print(f'Bank using fintech threshold ({best_threshold_b:.2f}):')
print(f'  Missed fraud: {cm_a_wrong["fn"]} (vs {cm_a["fn"]} with optimal)')
print(f'  Total cost:   EUR {cost_a_wrong:,.0f} (vs EUR {cost_a:,.0f} with optimal)')
print(f'  Extra cost:   EUR {cost_a_wrong - cost_a:,.0f} ({(cost_a_wrong - cost_a) / cost_a:.0%} increase)\n')

# Fintech using bank threshold (too low)
y_pred_b_wrong = (y_scores >= best_threshold_a).astype(int)
cost_b_wrong, cm_b_wrong = compute_total_cost(y_test, y_pred_b_wrong, COST_FN_B, COST_FP_B, COST_TP_B)
print(f'Fintech using bank threshold ({best_threshold_a:.2f}):')
print(f'  False declines: {cm_b_wrong["fp"]:,} (vs {cm_b["fp"]:,} with optimal)')
print(f'  Total cost:     EUR {cost_b_wrong:,.0f} (vs EUR {cost_b:,.0f} with optimal)')
print(f'  Extra cost:     EUR {cost_b_wrong - cost_b:,.0f} ({(cost_b_wrong - cost_b) / cost_b:.0%} increase)')

---

## Interview Talking Points

### How do you choose a threshold?
Never use the default 0.5. Define the business costs of each error type (false positive vs false negative),
then sweep thresholds and pick the one that minimizes total expected cost. If you can't get precise cost
numbers, use F-beta scores: F2 to favor recall, F0.5 to favor precision.

### Same model, different thresholds — why?
The model's probability output is fixed. The threshold is a business decision layered on top.
Two companies using the same model could operate at completely different thresholds because their
cost structures differ. This is why model deployment isn't just about accuracy — it's about
aligning the decision boundary with business objectives.

### What if costs change over time?
Fraud patterns shift, support costs change, customer LTV evolves. The threshold should be
re-evaluated periodically. In our FastAPI setup, the `PUT /threshold` endpoint lets you
update the threshold at runtime without redeploying the model.

### Why not just train separate models?
You could, but threshold tuning is cheaper and faster. Training is expensive (data, compute, validation).
Threshold tuning is a single parameter change on an already-validated model. Start with threshold tuning;
only retrain if the model's ranking ability (AUPRC) is insufficient.